# Прогнозирование летального исхода после COVID-19 (исправленная версия)

Курсовая работа. Столбцы выбираются по буквам Excel с проверкой заголовков (в первой версии все признаки были сдвинуты на один столбец из-за `set_index('N')`).

Перед запуском загрузите файл `Secondgroup-40-66 (2) исправленные (2).xlsx` в среду Colab.

In [ ]:
!pip install -q umap-learn shap


## Импорт библиотек и загрузка данных

In [ ]:
import re, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from openpyxl.utils import column_index_from_string as ci
warnings.filterwarnings('ignore')
RS = 42                                     # фиксированное зерно для воспроизводимости

PATH = 'Secondgroup-40-66 (2) исправленные (2).xlsx'
sheet = pd.read_excel(PATH, sheet_name='База общая', header=0)
assert str(sheet.columns[0]) == 'N'        # столбец A – номер записи
sheet = sheet[pd.to_numeric(sheet['N'], errors='coerce').notna()].copy()
sheet['N'] = sheet['N'].astype(int)
sheet = sheet.drop_duplicates(subset='N').reset_index(drop=True)
print('Пациентов:', len(sheet))

def norm(s):
    return re.sub(r'\s+', ' ', str(s)).strip().lower()

def col(letter, expect=None):
    """Числовой столбец по букве Excel (A = N) с проверкой заголовка.
    Индекс НЕ сдвигается: set_index не используется."""
    j = ci(letter) - 1
    if expect is not None:
        assert norm(expect) in norm(sheet.columns[j]), (letter, sheet.columns[j])
    return pd.to_numeric(sheet.iloc[:, j], errors='coerce')


## Определение популяции и целевого исхода

In [ ]:
M   = col('M', 'смерть в течении ковида') == 1     # смерть во время COVID-19
Nf  = col('N', 'смерть после ковида') == 1         # смерть после COVID-19
per = pd.concat([col('ACJ', 'в течение 1-го года'), col('ACK', 'через 1 год'),
                 col('ACL', 'через 2 года'), col('ACM', 'через 3 года')], axis=1)
any_period = (per == 1).any(axis=1)                 # логическое ИЛИ по периодам

confirmed  = any_period & Nf                        # смерть подтверждена двумя источниками
unresolved = (Nf & ~any_period) | (any_period & ~Nf)
population = ~M                                     # пережили острый COVID-19
use = population & ~unresolved                      # основная выборка

y = confirmed[use].astype(int).reset_index(drop=True)
y_sens = (any_period | Nf)[population].astype(int).reset_index(drop=True)
print('Популяция:', population.sum(), '| спорных записей:', (population & unresolved).sum())
print(f'Основная выборка: {len(y)} | смертей: {y.sum()} ({100 * y.mean():.1f} %)')


## Формирование наборов признаков FS1, FS2, FS3

In [ ]:
def anyof(letters):                       # «есть хотя бы одна стадия/форма заболевания»
    return pd.concat([col(l) for l in letters], axis=1).fillna(0).max(axis=1)

def build(mask):
    S = lambda s: s[mask].reset_index(drop=True)
    FS1 = pd.DataFrame({'age': S(col('D', 'возраст')),
                        'male': S(col('B', 'пол, мужской').fillna(0))})
    comorb = pd.DataFrame({
        'smoking': S(col('W', 'Курение').fillna(0)),
        'hypertension': S(anyof(['X', 'Y', 'Z', 'AA', 'AB', 'AC'])),
        'risk_veryhigh': S(col('AG', 'Риск очень высокий').fillna(0)),
        'CHF': S(anyof(['AH', 'AI', 'AJ', 'AK', 'AP', 'AQ', 'AR', 'AS'])),
        'IHD': S(col('AO', 'ИБС').fillna(0)),
        'angina': S(col('AT', 'Стенокардия').fillna(0)),
        'extrasystole': S(col('AY', 'Экстрасистолия').fillna(0)),
        'AF': S(col('BA', 'фебреляция предсердий').fillna(0)),
        'MI_history': S(anyof(['BI', 'BJ'])),
        'obesity': S(anyof(['BM', 'BN', 'BO', 'BP'])),
        'diabetes2': S(col('BR', 'Сахарный диабет 2 типа').fillna(0)),
        'IGT': S(col('BT', 'толеррантности').fillna(0)),
        'stroke': S(col('BV', 'ОНМК перенесенный').fillna(0)),
        'atheroscl': S(anyof(['BX', 'BY'])),
        'COPD': S(col('CG', 'ХОБЛ').fillna(0)),
        'emphysema': S(col('CH', 'эмфизема').fillna(0)),
        'asthma': S(col('CJ', 'Бронхиальная астма').fillna(0)),
        'CKD': S(col('CR', 'ХБП').fillna(0))})
    FS2 = pd.concat([FS1, comorb], axis=1)

    clin = pd.DataFrame({                 # пропуски НЕ заменяются нулём
        'fever': S(col('DJ', 'температура').fillna(0)),     # флаг, а не градусы
        'rr': S(col('DK', 'чд')), 'pulse': S(col('DL', 'пульс')),
        'hr': S(col('DM', 'чсс')), 'spo2': S(col('DN', 'SpO2')),
        'sbp': S(col('DO', 'САД')), 'dbp': S(col('DP', 'ДАД')),
        'wbc': S(col('FJ', 'WBC')), 'lymph': S(col('FK', 'LYM')),
        'hgb': S(col('FO', 'HGB')), 'plt': S(col('FQ', 'PLT')),
        'inr': S(col('FW', 'МНО')), 'fibr': S(col('FX', 'ФИБ')),
        'glucose': S(col('GC', 'глюкоза')), 'urea': S(col('GE', 'мочевина')),
        'creat': S(col('GF', 'креатинин')), 'ast': S(col('GL', 'AST')),
        'alt': S(col('GM', 'ALT')), 'crp': S(col('GO', 'RP мг/л'))})
    FS3 = pd.concat([FS2, clin], axis=1)
    return FS1, FS2, FS3

FS1, FS2, FS3 = build(use)
FS1s, FS2s, FS3s = build(population)      # для анализа чувствительности
FS = {'FS1': FS1, 'FS2': FS2, 'FS3': FS3}
for k, X in FS.items():
    print(k, X.shape[1], 'признаков, пропусков', round(100 * X.isna().mean().mean(), 1), '%')


## Разбиение выборки и конвейеры моделей

In [ ]:
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_validate, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

idx = np.arange(len(y))
i_tr, i_te = train_test_split(idx, test_size=0.2, stratify=y, random_state=RS)
y_tr, y_te = y.iloc[i_tr].reset_index(drop=True), y.iloc[i_te].reset_index(drop=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
spw = float((y_tr == 0).sum() / (y_tr == 1).sum())     # вес класса для XGBoost
print(f'train: {len(i_tr)} ({y_tr.sum()} смертей) | test: {len(i_te)} ({y_te.sum()} смертей)')

def make_pipe(model, scale=True):
    """Импутация медианой и масштабирование обучаются ТОЛЬКО на обучающем фолде."""
    steps = [('imp', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('sc', StandardScaler()))
    steps.append(('m', model))
    return Pipeline(steps)


## Сетки гиперпараметров и поиск по кросс-валидации

In [ ]:
def grids(spw=spw):
    return {
     'Dummy': (make_pipe(DummyClassifier(strategy='prior'), False), {}),
     'LogReg': (make_pipe(LogisticRegression(max_iter=5000, class_weight='balanced')),
                {'m__C': [0.01, 0.1, 1, 10]}),
     'DecisionTree': (make_pipe(DecisionTreeClassifier(class_weight='balanced',
                                random_state=RS), False),
                {'m__max_depth': [2, 3, 5], 'm__min_samples_leaf': [5, 10, 20]}),
     'RandomForest': (make_pipe(RandomForestClassifier(class_weight='balanced',
                                random_state=RS, n_estimators=400), False),
                {'m__max_depth': [2, 3, 5, None], 'm__min_samples_leaf': [1, 5],
                 'm__max_features': ['sqrt', 0.5]}),
     'XGBoost': (make_pipe(XGBClassifier(random_state=RS, eval_metric='logloss',
                                         scale_pos_weight=spw), False),
                {'m__max_depth': [2, 3, 4], 'm__learning_rate': [0.03, 0.1],
                 'm__n_estimators': [100, 200], 'm__subsample': [0.8, 1.0]})}

rows, models = [], {}
for fs, X in FS.items():
    Xtr = X.iloc[i_tr].reset_index(drop=True)
    for name, (pipe, grid) in grids().items():
        gs = GridSearchCV(pipe, grid, cv=cv, scoring='average_precision', n_jobs=-1)
        gs.fit(Xtr, y_tr)
        r = cross_validate(gs.best_estimator_, Xtr, y_tr, cv=cv,
                           scoring=['average_precision', 'roc_auc'])
        models[(fs, name)] = gs.best_estimator_
        rows.append(dict(fs=fs, model=name, params=gs.best_params_,
                         AP=r['test_average_precision'].mean(),
                         AP_sd=r['test_average_precision'].std(),
                         AUC=r['test_roc_auc'].mean(), AUC_sd=r['test_roc_auc'].std()))
cvdf = pd.DataFrame(rows)
print(cvdf.drop(columns='params').round(3).to_string(index=False))


## Однократная оценка на отложенной выборке

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, brier_score_loss, balanced_accuracy_score, confusion_matrix)

best_row = cvdf[cvdf.model != 'Dummy'].sort_values('AP', ascending=False).iloc[0]
best_fs, best_name = best_row.fs, best_row.model
Xb = FS[best_fs]
Xtr_b = Xb.iloc[i_tr].reset_index(drop=True)
Xte_b = Xb.iloc[i_te].reset_index(drop=True)
print('Выбрано по CV:', best_fs, best_name)

def metrics(p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    return dict(ROC_AUC=roc_auc_score(y_te, p), AP=average_precision_score(y_te, p),
                recall=recall_score(y_te, pred, zero_division=0),
                precision=precision_score(y_te, pred, zero_division=0),
                F1=f1_score(y_te, pred, zero_division=0), specificity=tn / (tn + fp),
                bal_acc=balanced_accuracy_score(y_te, pred),
                Brier=brier_score_loss(y_te, p), TN=tn, FP=fp, FN=fn, TP=tp)

test = pd.DataFrame([dict(model=n, **metrics(models[(best_fs, n)].predict_proba(Xte_b)[:, 1]))
                     for n in ['Dummy', 'LogReg', 'DecisionTree', 'RandomForest', 'XGBoost']])
print(test.round(3).to_string(index=False))


## Выбор порога и бутстрэп-интервалы

In [ ]:
best = models[(best_fs, best_name)]
# порог выбирается по OOF-вероятностям ТОЛЬКО на обучающей выборке
oof = cross_val_predict(best, Xtr_b, y_tr, cv=cv, method='predict_proba')[:, 1]
ths = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_tr, (oof >= t).astype(int), zero_division=0) for t in ths]
thr = ths[int(np.argmax(f1s))]
print(f'Порог по OOF: {thr:.2f}; F1(OOF) = {max(f1s):.3f}')
print(metrics(best.predict_proba(Xte_b)[:, 1], thr))

rng = np.random.default_rng(RS)
def boot(p, B=2000):
    """Стратифицированный бутстрэп 95 % ДИ для ROC-AUC и AP на тесте."""
    pos, neg = np.where(y_te == 1)[0], np.where(y_te == 0)[0]
    a, b = [], []
    for _ in range(B):
        s = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]
        a.append(roc_auc_score(y_te.values[s], p[s]))
        b.append(average_precision_score(y_te.values[s], p[s]))
    return np.percentile(a, [2.5, 97.5]).round(2), np.percentile(b, [2.5, 97.5]).round(2)

for n in ['LogReg', 'RandomForest', 'XGBoost', 'DecisionTree']:
    print(n, boot(models[(best_fs, n)].predict_proba(Xte_b)[:, 1]))


## ROC-, PR-кривые и калибровка

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for n in ['LogReg', 'DecisionTree', 'RandomForest', 'XGBoost']:
    p = models[(best_fs, n)].predict_proba(Xte_b)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, p)
    pr, rc, _ = precision_recall_curve(y_te, p)
    pt, pp = calibration_curve(y_te, p, n_bins=5, strategy='quantile')
    ax[0].plot(fpr, tpr, label=f'{n} (AUC = {roc_auc_score(y_te, p):.2f})')
    ax[1].plot(rc, pr, label=f'{n} (AP = {average_precision_score(y_te, p):.2f})')
    ax[2].plot(pp, pt, marker='s', label=n)
ax[0].plot([0, 1], [0, 1], ':'); ax[1].axhline(y_te.mean(), ls=':')
ax[2].plot([0, 1], [0, 1], 'k:')
for a, t in zip(ax, ['ROC', 'Precision-Recall', 'Калибровка']):
    a.set_title(t); a.legend()
plt.tight_layout(); plt.show()


## Вложенная повторная кросс-валидация 5×5

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RS)
inner = StratifiedKFold(n_splits=4, shuffle=True, random_state=RS)
spw_all = float((y == 0).sum() / (y == 1).sum())
nested = []
for fs, X in FS.items():
    for name, (pipe, grid) in grids(spw_all).items():
        if name == 'RandomForest':
            pipe.set_params(m__n_estimators=200)
        est = GridSearchCV(pipe, grid, cv=inner, scoring='average_precision') if grid else pipe
        r = cross_validate(est, X, y, cv=outer, n_jobs=-1,
                           scoring=['average_precision', 'roc_auc', 'neg_brier_score'])
        nested.append(dict(fs=fs, model=name,
                           AP=r['test_average_precision'].mean(),
                           AP_sd=r['test_average_precision'].std(),
                           AUC=r['test_roc_auc'].mean(), AUC_sd=r['test_roc_auc'].std(),
                           Brier=-r['test_neg_brier_score'].mean()))
nested = pd.DataFrame(nested)
print(nested.round(3).to_string(index=False))


## Снижение размерности: PCA

In [ ]:
from sklearn.decomposition import PCA
X3tr = FS3.iloc[i_tr].reset_index(drop=True); X3te = FS3.iloc[i_te].reset_index(drop=True)
prep = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]).fit(X3tr)
Z = prep.transform(X3tr)
pca = PCA().fit(Z)                                   # обучается только на train
evr = pca.explained_variance_ratio_
print('PC1, PC2:', evr[:2].round(3), '| компонент для 95 %:', np.searchsorted(np.cumsum(evr), .95) + 1)

def lr_pipe(extra=None):
    st = [('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]
    if extra: st.append(extra)
    st.append(('m', LogisticRegression(max_iter=5000, class_weight='balanced')))
    return Pipeline(st)

dr = []
for label, pipe in [('исходные', lr_pipe()),
                    ('PCA 95 %', lr_pipe(('pca', PCA(n_components=0.95, svd_solver='full')))),
                    ('PCA 10', lr_pipe(('pca', PCA(n_components=10)))),
                    ('PCA 2', lr_pipe(('pca', PCA(n_components=2))))]:
    gs = GridSearchCV(pipe, {'m__C': [0.01, 0.1, 1, 10]}, cv=cv,
                      scoring='average_precision').fit(X3tr, y_tr)
    r = cross_validate(gs.best_estimator_, X3tr, y_tr, cv=cv, scoring=['average_precision', 'roc_auc'])
    p = gs.best_estimator_.predict_proba(X3te)[:, 1]
    dr.append(dict(method=label, CV_AP=r['test_average_precision'].mean(),
                   CV_AUC=r['test_roc_auc'].mean(), test_AUC=roc_auc_score(y_te, p),
                   test_AP=average_precision_score(y_te, p)))
print(pd.DataFrame(dr).round(3).to_string(index=False))


## Снижение размерности: UMAP внутри фолдов

In [ ]:
# В работе использована реализация umap-js 1.4.0 (вызов из Python через Node.js);
# в Colab можно применить umap-learn – значения будут близкими, но не идентичными.
try:
    import umap
    aps = []
    for a, b in cv.split(X3tr, y_tr):
        pf = Pipeline([('imp', SimpleImputer(strategy='median')),
                       ('sc', StandardScaler())]).fit(X3tr.iloc[a])
        um = umap.UMAP(n_components=5, n_neighbors=15, min_dist=0.1, random_state=RS)
        e_tr = um.fit_transform(pf.transform(X3tr.iloc[a]))   # UMAP учится на фолде
        e_va = um.transform(pf.transform(X3tr.iloc[b]))
        m = Pipeline([('sc', StandardScaler()), ('m', LogisticRegression(
            max_iter=5000, class_weight='balanced'))]).fit(e_tr, y_tr.iloc[a])
        aps.append(average_precision_score(y_tr.iloc[b], m.predict_proba(e_va)[:, 1]))
    print('UMAP(5) + LogReg, CV AP =', round(np.mean(aps), 3))
    Zu = umap.UMAP(n_components=2, random_state=RS).fit_transform(Z)
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    for k, (E, t) in enumerate([(pca.transform(Z)[:, :2], 'PCA'), (Zu, 'UMAP')]):
        for c, lab in [(0, 'выжил'), (1, 'умер')]:
            ax[k].scatter(E[y_tr == c, 0], E[y_tr == c, 1], s=25, label=lab)
        ax[k].set_title(t); ax[k].legend()
    plt.show()
except ImportError:
    print('pip install umap-learn')


## Интерпретация: коэффициенты и перестановочная важность

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.base import clone
lr3 = models[('FS3', 'LogReg')]
coef = pd.Series(lr3.named_steps['m'].coef_[0], index=FS3.columns).sort_values()
coef.plot.barh(figsize=(7, 9)); plt.title('Коэффициенты LogReg (стандартизированная шкала)')
plt.tight_layout(); plt.show()

# важность считается на валидационных фолдах обучающей выборки, а не на тесте
pis = []
for a, b in cv.split(Xtr_b, y_tr):
    mdl = clone(best).fit(Xtr_b.iloc[a], y_tr.iloc[a])
    pi = permutation_importance(mdl, Xtr_b.iloc[b], y_tr.iloc[b], n_repeats=20,
                                random_state=RS, scoring='average_precision')
    pis.append(pi.importances_mean)
imp = pd.Series(np.mean(pis, axis=0), index=Xb.columns).sort_values(ascending=False)
imp.head(15)[::-1].plot.barh(figsize=(7, 5)); plt.title('Permutation importance (CV)')
plt.tight_layout(); plt.show()


## Интерпретация: SHAP и частичная зависимость

In [ ]:
import xgboost as xgb
from sklearn.inspection import PartialDependenceDisplay
xgb_pipe = models[(best_fs, 'XGBoost')]
Xi = xgb_pipe.named_steps['imp'].transform(Xb)
booster = xgb_pipe.named_steps['m'].get_booster()
# TreeSHAP встроен в XGBoost: последний столбец – базовое значение
sv = booster.predict(xgb.DMatrix(Xi, feature_names=list(Xb.columns)), pred_contribs=True)[:, :-1]
shap_imp = pd.Series(np.abs(sv).mean(0), index=Xb.columns).sort_values(ascending=False)
print(shap_imp.head(10).round(3))
try:
    import shap                                     # при наличии библиотеки shap
    shap.summary_plot(sv, pd.DataFrame(Xi, columns=Xb.columns), max_display=15)
except ImportError:
    pass

feats = [f for f in shap_imp.index if Xb[f].nunique() > 2][:4]
PartialDependenceDisplay.from_estimator(best, Xtr_b, feats, kind='average')
plt.tight_layout(); plt.show()
